# ML Security Experiments: Pipelines + Ensembles

**Google Colab lab notebook** — two complete projects in one notebook:

1. **ML-Based Ad URL Classification:** `0 = ALLOW`, `1 = BLOCK`, using TF-IDF text features, URL signals, Logistic Regression, Naive Bayes, SVM, and a soft-voting ensemble.
2. **Keystroke Dynamics Identification:** identify one of five users from *precomputed typing-timing features* using scaling/imputation, five ML classifiers, soft voting, and stacking.

**Workflow:** Upload CSV files → read and inspect → train/test split → Pipeline → cross-validation → pick model using *training only* → held-out evaluation → demo predictions → save results → publish code to GitHub.

> Important: The first model classifies URLs; it **does not actually block browser requests**. The second dataset is not accurate enough for real authentication. Do not interpret small test-set scores as production quality.

## 0. Install dependencies
Run this cell in a fresh **Google Colab** runtime. Colab may already have these packages installed.

In [ ]:
%pip -q install numpy pandas scikit-learn matplotlib joblib

## 1. Import the Python libraries

In [ ]:
from pathlib import Path
import json
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import (
    ExtraTreesClassifier, RandomForestClassifier,
    StackingClassifier, VotingClassifier,
)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, classification_report,
    ConfusionMatrixDisplay, confusion_matrix,
)
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.naive_bayes import ComplementNB, GaussianNB
from sklearn.pipeline import FeatureUnion, Pipeline, make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

RANDOM_SEED = 42
DATA_DIR = Path("data")
RESULTS_DIR = Path("results")
DATA_DIR.mkdir(exist_ok=True)
RESULTS_DIR.mkdir(exist_ok=True)
print("Libraries imported successfully.")

## 2. Upload your datasets

In Colab, click **Choose Files** and select these *two CSV files*:

- `ad_urls (1).csv` (or `ad_urls.csv`)
- `keystroke_features (1).csv` (or `keystroke_features.csv`)

If you downloaded and extracted the complete project locally, the `data/` folder already contains them, so this cell skips uploading.

In [ ]:
AD_NAMES = ("ad_urls.csv", "ad_urls (1).csv")
KEY_NAMES = ("keystroke_features.csv", "keystroke_features (1).csv")

def csv_path(names):
    for directory in (DATA_DIR, Path(".")):
        for name in names:
            candidate = directory / name
            if candidate.is_file():
                return candidate
    return None

if csv_path(AD_NAMES) is None or csv_path(KEY_NAMES) is None:
    try:
        from google.colab import files
    except ImportError:
        print("Not in Colab: copy both CSVs into data/ and rerun this cell.")
    else:
        print("Please upload both CSVs. They stay in this Colab runtime.")
        uploaded = files.upload()
        for name, content in uploaded.items():
            target = DATA_DIR / Path(name).name  # Never write outside data/
            target.write_bytes(content)
            print("Uploaded:", target)
else:
    print("Both datasets are already available in data/ or the working directory.")

## 3. Read the CSV files using Pandas
The following code supports both the original uploaded names and the shorter names in this Git-ready project.

In [ ]:
ad_path = csv_path(AD_NAMES)
key_path = csv_path(KEY_NAMES)
if ad_path is None or key_path is None:
    raise FileNotFoundError("Upload both CSVs in Step 2, then run this cell again.")

ad_df = pd.read_csv(ad_path)
key_df = pd.read_csv(key_path)

print(f"Ad dataset: {ad_path}, shape={ad_df.shape}")
display(ad_df.head())
print(f"Keystroke dataset: {key_path}, shape={key_df.shape}")
display(key_df.head())

## 4. Validate data and explore labels
We keep the two problems separate: URLs are text, whereas keystroke features are numeric.

In [ ]:
required_ad = {"url", "label"}
if not required_ad.issubset(ad_df.columns):
    raise ValueError(f"Ad CSV needs columns {required_ad}.")
ad_df = ad_df.dropna(subset=["url", "label"]).copy()
ad_df["url"] = ad_df["url"].astype(str).str.strip()
ad_df["label"] = pd.to_numeric(ad_df["label"], errors="raise").astype(int)
ad_df = ad_df[ad_df["url"].ne("")].drop_duplicates(subset=["url"])
if not set(ad_df["label"].unique()).issubset({0, 1}):
    raise ValueError("Ad labels must be 0=ALLOW or 1=BLOCK.")
if ad_df["label"].nunique() != 2:
    raise ValueError("Ad dataset must contain both classes.")

if "user_id" not in key_df.columns:
    raise ValueError("Keystroke CSV must include 'user_id'.")
key_df = key_df.dropna(subset=["user_id"]).copy()
feature_names = key_df.columns.drop("user_id").tolist()
if not feature_names:
    raise ValueError("Keystroke CSV requires numeric feature columns.")
key_df[feature_names] = key_df[feature_names].apply(pd.to_numeric, errors="coerce")
if key_df[feature_names].isna().all(axis=0).any():
    raise ValueError("At least one keystroke feature has no numeric values.")

print("Ad labels (0=ALLOW, 1=BLOCK):")
print(ad_df["label"].value_counts().sort_index().to_string())
print("Keystroke samples per user:")
print(key_df["user_id"].value_counts().sort_index().to_string())
print("Keystroke input features:", feature_names)

---
## Experiment 1 — ML-Based Ad URL Classification

### 5. Split into training and test data
The **test data stays unseen** until after model selection. This 30-URL dataset is very small, so scores are demonstrations only.

In [ ]:
X_ad_train, X_ad_test, y_ad_train, y_ad_test = train_test_split(
    ad_df["url"], ad_df["label"],
    test_size=0.30, stratify=ad_df["label"], random_state=RANDOM_SEED,
)
print(f"Training URLs: {len(X_ad_train)}; held-out test URLs: {len(X_ad_test)}")

### 6. Build an ML Pipeline for URL features
A `FeatureUnion` extracts character n-grams, URL word n-grams, and simple ad/tracker indicators. The features are fitted **inside the Pipeline**, preventing training/test feature leakage.

In [ ]:
class URLSignals(BaseEstimator, TransformerMixin):
    """Three fixed, interpretable ad/tracker lexical indicators."""
    patterns = (
        r"(?:^|[^a-z])(?:ads?|adserver|advertisements?|doubleclick|banner)(?:[^a-z]|$)",
        r"track|analytics|pixel|beacon|retarget|impression|collect",
        r"sponsor|promot|affiliate|marketing|campaign|click",
    )

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        return np.asarray([
            [float(bool(re.search(p, str(url).lower()))) for p in self.patterns]
            for url in X
        ], dtype=float)


def ad_pipeline(classifier):
    features = FeatureUnion(
        transformer_list=[
            ("character_tfidf", TfidfVectorizer(analyzer="char", ngram_range=(3, 5))),
            ("word_tfidf", TfidfVectorizer(token_pattern=r"[a-zA-Z]+", ngram_range=(1, 2))),
            ("url_signals", URLSignals()),
        ],
        transformer_weights={"url_signals": 3.0},
    )
    return Pipeline([("features", features), ("classifier", classifier)])

print("URL Pipeline created: TF-IDF + URLSignals + classifier")

### 7. Compare algorithms and combine three models
The soft-voting ensemble averages class probabilities from **Logistic Regression + Complement Naive Bayes + Linear SVM**. Select the best candidate using **5-fold CV on training data only**, never using the final test set to choose a model.

In [ ]:
def compare_by_cv(models, X, y, *, folds=5):
    cv = StratifiedKFold(n_splits=folds, shuffle=True, random_state=7)
    rows = []
    for name, model in models.items():
        scores = cross_val_score(model, X, y, cv=cv, scoring="accuracy", n_jobs=1)
        rows.append({"model": name, "cv_mean": scores.mean(), "cv_std": scores.std()})
    return (pd.DataFrame(rows)
            .sort_values(["cv_mean", "cv_std"], ascending=[False, True], kind="stable")
            .reset_index(drop=True))

ad_lr = LogisticRegression(C=1.0, max_iter=1000)
ad_nb = ComplementNB(alpha=0.5)
ad_svm = SVC(kernel="linear", C=1.0, probability=True, random_state=RANDOM_SEED)
ad_voting = VotingClassifier(
    estimators=[("lr", clone(ad_lr)), ("nb", clone(ad_nb)), ("svm", clone(ad_svm))],
    voting="soft",
)
ad_models = {
    "Logistic Regression": ad_pipeline(ad_lr),
    "Complement Naive Bayes": ad_pipeline(ad_nb),
    "Linear SVM": ad_pipeline(ad_svm),
    "Soft Voting Ensemble": ad_pipeline(ad_voting),
}
ad_cv = compare_by_cv(ad_models, X_ad_train, y_ad_train)
display(ad_cv.style.format({"cv_mean": "{:.1%}", "cv_std": "{:.1%}"}))
selected_ad_name = str(ad_cv.loc[0, "model"])
print("Selected from training CV:", selected_ad_name)

### 8. Evaluate the chosen URL model on unseen URLs
This test set contains only 9 URLs; even 9/9 correct is **not** evidence of 100% real-world blocking accuracy. Note that URL signal rules were developed while exploring this small teaching dataset.

In [ ]:
ad_best = clone(ad_models[selected_ad_name]).fit(X_ad_train, y_ad_train)
ad_predictions = ad_best.predict(X_ad_test)
ad_accuracy = accuracy_score(y_ad_test, ad_predictions)
print(f"Holdout test accuracy: {ad_accuracy:.1%} ({(ad_predictions == y_ad_test).sum()}/{len(y_ad_test)})")
print(classification_report(y_ad_test, ad_predictions, labels=[0, 1],
                            target_names=["ALLOW", "BLOCK"], zero_division=0))

ConfusionMatrixDisplay.from_predictions(
    y_ad_test, ad_predictions, labels=[0, 1], display_labels=["ALLOW", "BLOCK"],
    cmap="Blues", colorbar=False,
)
plt.title("Ad classifier — held-out test")
plt.tight_layout()
plt.show()

sample_urls = [
    "https://ads.example.net/banner/new.js",
    "https://university.example.edu/students",
    "https://doubleclick.net/pagead/ads?slot=1",
]
for url, prediction in zip(sample_urls, ad_best.predict(sample_urls)):
    print(("BLOCK" if prediction == 1 else "ALLOW"), "|", url)
print("These are predictions; no browser/network requests were blocked.")

---
## Experiment 2 — ML-Based Keystroke Dynamics

### 9. Train/test split of typing-timing features
We predict `user_id` using eight precomputed timing features. No plaintext keystrokes or passwords are collected by this notebook.

In [ ]:
X_key = key_df[feature_names]
y_key = key_df["user_id"].astype(str)
X_key_train, X_key_test, y_key_train, y_key_test = train_test_split(
    X_key, y_key, test_size=0.25, stratify=y_key, random_state=RANDOM_SEED,
)
print(f"Train samples: {len(X_key_train)}; test samples: {len(X_key_test)}")
print("Users:", sorted(y_key.unique()))

### 10. Create Pipelines and Voting/Stacking Ensembles

- Logistic Regression, RBF SVM, and GaussianNB: `SimpleImputer → StandardScaler → Model`.
- Random Forest / Extra Trees: `SimpleImputer → Model`.
- Voting averages probabilities; stacking fits a meta-classifier using out-of-fold training predictions.

In [ ]:
def scaled(model):
    return make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), model)

def trees(model):
    return make_pipeline(SimpleImputer(strategy="median"), model)

key_lr = scaled(LogisticRegression(C=1, max_iter=1000))
key_svm = scaled(SVC(C=1, kernel="rbf", probability=True, random_state=RANDOM_SEED))
key_nb = scaled(GaussianNB())
key_rf = trees(RandomForestClassifier(n_estimators=180, min_samples_leaf=3,
                                      random_state=RANDOM_SEED, n_jobs=1))
key_et = trees(ExtraTreesClassifier(n_estimators=180, min_samples_leaf=3,
                                    random_state=RANDOM_SEED, n_jobs=1))

def key_members():
    return [("lr", clone(key_lr)), ("svm", clone(key_svm)), ("extra", clone(key_et))]

key_models = {
    "Logistic Regression": key_lr,
    "RBF SVM": key_svm,
    "Gaussian Naive Bayes": key_nb,
    "Random Forest": key_rf,
    "Extra Trees": key_et,
    "Soft Voting Ensemble": VotingClassifier(key_members(), voting="soft"),
    "Stacking Ensemble": StackingClassifier(
        key_members(), final_estimator=LogisticRegression(max_iter=1000),
        cv=3, stack_method="predict_proba",
    ),
}
print("Created", len(key_models), "Pipeline/classifier candidates.")

### 11. Cross-validation: compare *all* models fairly
Compare all single models and both ensembles. A more complex ensemble is **not necessarily more accurate**. The best model is selected using training-set CV results.

In [ ]:
key_cv = compare_by_cv(key_models, X_key_train, y_key_train)
display(key_cv.style.format({"cv_mean": "{:.1%}", "cv_std": "{:.1%}"}))
selected_key_name = str(key_cv.loc[0, "model"])
ensemble_names = ["Soft Voting Ensemble", "Stacking Ensemble"]
best_ensemble_name = str(
    key_cv[key_cv["model"].isin(ensemble_names)].iloc[0]["model"]
)
print("Best model by training CV:", selected_key_name)
print("Best ensemble by training CV:", best_ensemble_name)

### 12. Evaluate on unseen keystroke samples
Since this experiment focuses on **combining models**, the main report uses the **best ensemble by training CV**. For transparency, we also evaluate the best CV-ranked candidate from all models. We never use the test scores to pick a winner.

In [ ]:
key_best = clone(key_models[best_ensemble_name]).fit(X_key_train, y_key_train)
key_pred = key_best.predict(X_key_test)
key_accuracy = accuracy_score(y_key_test, key_pred)
cv_top_model = clone(key_models[selected_key_name]).fit(X_key_train, y_key_train)
cv_top_accuracy = accuracy_score(y_key_test, cv_top_model.predict(X_key_test))
key_baseline = DummyClassifier(strategy="most_frequent").fit(X_key_train, y_key_train)
key_baseline_accuracy = accuracy_score(y_key_test, key_baseline.predict(X_key_test))

print(f"Best ensemble ({best_ensemble_name}) holdout accuracy: {key_accuracy:.1%}")
print(f"Top CV-ranked candidate ({selected_key_name}) holdout accuracy: {cv_top_accuracy:.1%}")
print(f"Majority-class baseline accuracy: {key_baseline_accuracy:.1%}")
print("\nPer-user report for selected ensemble:")
print(classification_report(y_key_test, key_pred, zero_division=0))

labels = sorted(y_key.unique())
ConfusionMatrixDisplay.from_predictions(
    y_key_test, key_pred, labels=labels,
    display_labels=labels, xticks_rotation=35,
    cmap="Blues", colorbar=False,
)
plt.title("Keystroke user identification — held-out test")
plt.tight_layout()
plt.show()

print("Example: actual =", y_key_test.iloc[0],
      "| predicted =", key_best.predict(X_key_test.iloc[[0]])[0])
print("Near-chance accuracy is NOT suitable for authentication.")

## 13. Export evaluation results
The following files are created in `results/` for download or private storage. Generated predictions/metrics and user biometric data are not automatically committed to GitHub.

In [ ]:
ad_cv.to_csv(RESULTS_DIR / "ad_cv_results.csv", index=False)
key_cv.to_csv(RESULTS_DIR / "keystroke_cv_results.csv", index=False)
summary = {
    "ad_model": selected_ad_name,
    "ad_holdout_accuracy": float(ad_accuracy),
    "ad_test_samples": int(len(y_ad_test)),
    "keystroke_model": best_ensemble_name,
    "keystroke_holdout_accuracy": float(key_accuracy),
    "keystroke_cv_top_candidate": selected_key_name,
    "keystroke_cv_top_candidate_holdout_accuracy": float(cv_top_accuracy),
    "keystroke_baseline_accuracy": float(key_baseline_accuracy),
    "keystroke_test_samples": int(len(y_key_test)),
}
(RESULTS_DIR / "metrics.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
print("Saved:")
for file in sorted(RESULTS_DIR.glob("*")):
    if file.is_file():
        print(" -", file)
print("\nSummary:")
print(json.dumps(summary, indent=2))

## 14. GitHub: publish the code and README

The accompanying ZIP already contains the notebook, two runnable `.py` scripts, `README.md`, `requirements.txt`, `.gitignore`, and sample datasets.

**Recommended workflow**

1. Download/extract the project ZIP to your computer.
2. Create a **new empty GitHub repository** (for example, `ml-security-pipelines`), **without** asking GitHub to create another README.
3. Open a terminal inside the extracted project folder and run the commands below (replace placeholders).
4. Authenticate through your normal GitHub setup, such as Git Credential Manager or SSH. **Do not paste personal access tokens in notebooks.**
5. For later notebook changes in Colab, use **File → Save a copy in GitHub**, or download the updated `.ipynb` and commit it normally.

```bash
git init
git branch -M main
git add README.md requirements.txt .gitignore ML_Security_Pipelines_Ensembles_Colab.ipynb
git add ad_blocker_ensemble.py keystroke_ensemble.py data/ad_urls.csv
git commit -m "Add Colab ML security pipeline experiments"
git remote add origin https://github.com/YOUR_USERNAME/ml-security-pipelines.git
git push -u origin main
```

**Privacy:** The `.gitignore` excludes `data/keystroke_features.csv` by default because typing-timing data can be behavioral biometric information. Your downloaded ZIP includes it for running the experiment. Share it in a public repository only if authorized.

## 15. Limitations and good lab practices

- A **URL classifier** is not a functioning browser ad blocker; integration with a browser extension or filtering engine is a separate task.
- The ad dataset has only about **30 examples**; the holdout set has **9**, and the lexical features were created while exploring these examples. Test on independent domains before claiming generalization.
- The keystroke dataset has about **500 records for 5 users** and weak user separation; low accuracy is a property of the available features, not a reason to report only a favorable seed.
- For actual keystroke authentication: use consented genuine samples, separate sessions for train/test, and evaluate **false acceptance rate (FAR)**, **false rejection rate (FRR)** and **equal error rate (EER)**.
- Never use the held-out test score to tune features, select a new model, or claim production readiness.